# Estratégia 1 — Fama-French 5 Fatores (FF5)
**Competição de Carteira de Investimentos 2026 — PPGOLD/UFPR**

---
## O que é o modelo Fama-French 5 Fatores?

Proposto por Eugene Fama e Kenneth French (2015), o modelo FF5 expande o clássico CAPM acrescentando 4 fatores adicionais além do risco de mercado. É amplamente reconhecido na literatura acadêmica (Damodaran, Brigham — bibliografia do plano de ensino) como um dos frameworks mais robustos para explicar o retorno de ações em corte transversal.

| Fator | Proxy usada neste notebook | Lógica |
|---|---|---|
| **MKT** — Risco de Mercado | Beta vs. IBOVESPA (^BVSP) | Exposição ao risco sistêmico |
| **SMB** — Size (Tamanho) | Market Cap (P/VP × Patrim. Líq.) | Small caps tendem a superar |
| **HML** — Value | 1 / P/VP (Book-to-Market) | Ações baratas vs. growth |
| **RMW** — Profitability | ROE (Retorno sobre Patrimônio) | Empresas mais lucrativas superam |
| **CMA** — Investment | Inverso do Cresc. Receita 5a | Empresas conservadoras superam |

## Objetivo
Usar os 5 fatores como **sistema de pontuação** para ranquear toda a bolsa B3, selecionar os **Top 12 ativos mais atraentes** e, sobre eles, aplicar otimização combinatória de Markowitz (C(12,4) = 495 carteiras) para encontrar os **4 ativos com maior Índice de Sharpe**.


---
## Etapa 1 — Importação de Bibliotecas

In [ ]:
import requests, io, warnings
import pandas as pd
import numpy as np
import yfinance as yf
from itertools import combinations
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Bibliotecas carregadas com sucesso.')

---
## Etapa 2 — Web Scraping: Universo completo da B3 (Fundamentus)

Conectamos ao site Fundamentus e extraímos a tabela com todos os ativos listados na B3, incluindo métricas fundamentalistas prontas. Nenhuma ação é pré-selecionada manualmente — o modelo decide tudo.


In [ ]:
headers = {'User-Agent': 'Mozilla/5.0'}
url = 'https://www.fundamentus.com.br/resultado.php'
resposta = requests.get(url, headers=headers)
df_b3 = pd.read_html(io.StringIO(resposta.text), decimal=',', thousands='.')[0]

def pct_to_float(s):
    return pd.to_numeric(
        s.astype(str).str.replace('.','',regex=False)
          .str.replace(',','.',regex=False)
          .str.replace('%','',regex=False), errors='coerce') / 100.0

for col in ['ROE','ROIC','Mrg Ebit','Mrg. Líq.','Div.Yield','Cresc. Rec.5a']:
    df_b3[col] = pct_to_float(df_b3[col])

for col in ['P/L','P/VP','EV/EBITDA','Liq.2meses','Patrim. Líq','Dív.Brut/ Patrim.']:
    df_b3[col] = pd.to_numeric(df_b3[col], errors='coerce')

print(f'Total de ativos na B3 (Fundamentus): {len(df_b3)}')
df_b3.head(3)

---
## Etapa 3 — Filtro de Liquidez (Regra do Edital)

O edital proíbe ativos ilíquidos. Aplicamos cortes severos para garantir que apenas papéis negociados em volume robusto entrem no universo de análise.


In [ ]:
# Filtros de sobrevivência
df = df_b3[
    (df_b3['Liq.2meses'] > 5_000_000) &  # Volume bimestral > R$ 5 milhões
    (df_b3['P/L'] > 0) &                  # Empresa lucrativa (sem prejuízo)
    (df_b3['P/VP'] > 0) &                 # Patrimônio líquido positivo
    (df_b3['ROE'] > 0)                    # ROE positivo
].copy().reset_index(drop=True)

print(f'Ativos após filtro de liquidez e qualidade: {len(df)}')

---
## Etapa 4 — Cálculo do Beta vs. IBOVESPA (Fator MKT)

O beta mede a sensibilidade do ativo às variações do mercado. Beta > 1 → mais volátil que o índice; Beta < 1 → mais defensivo. Para a competição (superar CDI), queremos betas moderados — volatilidade controlada.


In [ ]:
start, end = '2023-01-01', '2026-03-26'
ibov = yf.download('^BVSP', start=start, end=end, progress=False)['Close'].pct_change().dropna()
ibov.name = 'IBOV'

tickers_sa = [t + '.SA' for t in df['Papel'].tolist()]
sample = tickers_sa[:80]  # Limitamos para performance; aumentar se necessário

precos = yf.download(sample, start=start, end=end, progress=False)['Close']
precos.columns = [c.replace('.SA','') for c in precos.columns]
retornos = precos.pct_change().dropna()

# Calcula beta para cada ticker disponível
betas = {}
ibov_aligned = ibov.reindex(retornos.index).dropna()
for col in retornos.columns:
    serie = retornos[col].reindex(ibov_aligned.index).dropna()
    if len(serie) > 100:
        cov = np.cov(serie, ibov_aligned)[0,1]
        var = np.var(ibov_aligned)
        betas[col] = cov / var if var != 0 else np.nan

df_beta = pd.DataFrame.from_dict(betas, orient='index', columns=['Beta'])
print(f'Beta calculado para {len(df_beta)} ativos.')

---
## Etapa 5 — Construção do Score FF5

Para cada fator, ranqueamos os ativos (ranking relativo 1 a N) e **invertemos** os fatores em que 'menor é melhor' (P/VP, Cresc. Receita). A soma dos 5 ranks gera o **Score FF5**: quanto menor, mais atrativo é o ativo segundo o modelo.


In [ ]:
# Faz merge beta no df principal
df = df[df['Papel'].isin(df_beta.index)].copy()
df['Beta'] = df['Papel'].map(betas)

# Market Cap aproximado
df['MktCap'] = df['Patrim. Líq'] * df['P/VP']

# Calcular proxies dos 5 fatores e seus rankings
df['rank_MKT'] = df['Beta'].abs().rank(ascending=True)          # Menor beta = mais defensivo
df['rank_SMB'] = df['MktCap'].rank(ascending=True)              # Small caps (menor mktcap)
df['rank_HML'] = (1 / df['P/VP']).rank(ascending=False)         # Maior B/M (menor P/VP)
df['rank_RMW'] = df['ROE'].rank(ascending=False)                # Maior ROE
df['rank_CMA'] = df['Cresc. Rec.5a'].rank(ascending=True)       # Menor crescimento invest. (conservador)

# Score total: soma dos ranks (menor = melhor)
df['Score_FF5'] = df[['rank_MKT','rank_SMB','rank_HML','rank_RMW','rank_CMA']].mean(axis=1)
df_ranked = df.sort_values('Score_FF5').dropna(subset=['Score_FF5'])

# Top 12
top12 = df_ranked.head(12)
print('Top 12 ativos selecionados pelo Score FF5:')
display(top12[['Papel','Beta','P/VP','ROE','Cresc. Rec.5a','Score_FF5']].reset_index(drop=True))

---
## Etapa 6 — Visualização das Métricas FF5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Top 12 Ativos — Métricas Fama-French 5 Fatores', fontsize=14, fontweight='bold')

top12.set_index('Papel')['ROE'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('ROE (Fator RMW)')
axes[0].set_xlabel('ROE')

top12.set_index('Papel')['P/VP'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('P/VP — Valor Book-to-Market (HML)')
axes[1].set_xlabel('P/VP')

top12.set_index('Papel')['Beta'].sort_values().plot(kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Beta vs. IBOVESPA (MKT)')
axes[2].set_xlabel('Beta')

plt.tight_layout()
plt.show()

---
## Etapa 7 — Otimização Combinatória de Markowitz: C(12,4) = 495 Carteiras

Com os 12 melhores ativos pelo modelo FF5, testamos **todas as 495 combinações únicas** de 4 ativos. Para cada combinação, o otimizador não-linear (SciPy/SLSQP) calcula os pesos que maximizam o **Índice de Sharpe** usando dados históricos de retorno e covariância (5 anos). A carteira vencedora é a que apresenta o maior Sharpe global.


In [ ]:
tickers_top = [t + '.SA' for t in top12['Papel'].tolist()]
rf = 0.1075  # CDI estimado como taxa livre de risco

# Dados históricos dos top 12
precos_top = yf.download(tickers_top, start='2021-03-26', end='2026-03-26', progress=False)['Close']
precos_top = precos_top.dropna(axis=1, thresh=int(len(precos_top)*0.8))
precos_top = precos_top.ffill().dropna()
tickers_validos = precos_top.columns.tolist()

ret_diario = precos_top.pct_change().dropna()
ret_anual = ret_diario.mean() * 252
cov_anual = ret_diario.cov() * 252

def neg_sharpe(w, mu, cov, rf):
    r = np.dot(w, mu)
    v = np.sqrt(w @ cov @ w)
    return -(r - rf) / v

melhor_sharpe = -np.inf
melhor_comb = None
melhores_pesos = None

for comb in combinations(tickers_validos, 4):
    mu = ret_anual[list(comb)].values
    cov = cov_anual.loc[list(comb), list(comb)].values
    w0 = np.array([0.25]*4)
    bounds = [(0,1)]*4
    constraints = {'type':'eq','fun': lambda w: w.sum()-1}
    res = minimize(neg_sharpe, w0, args=(mu,cov,rf), method='SLSQP', bounds=bounds, constraints=constraints)
    if -res.fun > melhor_sharpe:
        melhor_sharpe = -res.fun
        melhor_comb = list(comb)
        melhores_pesos = res.x

mu_f = ret_anual[melhor_comb].values
cov_f = cov_anual.loc[melhor_comb, melhor_comb].values
ret_final = np.dot(melhores_pesos, mu_f)
vol_final = np.sqrt(melhores_pesos @ cov_f @ melhores_pesos)

print('='*55)
print('CARTEIRA ÓTIMA — FAMA-FRENCH 5 FATORES')
print('='*55)
for t, w in zip(melhor_comb, melhores_pesos):
    print(f'  {t.replace(".SA","")}: {w*100:.2f}%')
print(f'\n  Retorno Esperado : {ret_final*100:.2f}% a.a.')
print(f'  Volatilidade     : {vol_final*100:.2f}% a.a.')
print(f'  Índice de Sharpe : {melhor_sharpe:.4f}')

---
## Etapa 8 — Visualização: Fronteira Eficiente e Portfólio Ótimo

In [ ]:
# Simula 10.000 portfólios aleatórios com os 4 ativos finais
mu_f = ret_anual[melhor_comb].values
cov_f = cov_anual.loc[melhor_comb, melhor_comb].values
n_sim = 10000
rets, vols, sharpes = [], [], []
for _ in range(n_sim):
    w = np.random.dirichlet(np.ones(4))
    r = np.dot(w, mu_f)
    v = np.sqrt(w @ cov_f @ w)
    rets.append(r); vols.append(v); sharpes.append((r-rf)/v)

plt.figure(figsize=(10,6))
sc = plt.scatter(vols, rets, c=sharpes, cmap='viridis', s=8, alpha=0.4)
plt.colorbar(sc, label='Índice de Sharpe')
plt.scatter(vol_final, ret_final, marker='*', color='red', s=300, zorder=5,
            label=f'Portfólio Ótimo FF5 (Sharpe={melhor_sharpe:.2f})')
plt.axhline(rf, color='gray', linestyle='--', label=f'CDI ({rf*100:.1f}%)')
plt.xlabel('Risco Anualizado (Volatilidade)')
plt.ylabel('Retorno Anualizado')
plt.title('Fronteira Eficiente — Estratégia Fama-French 5 Fatores')
plt.legend()
plt.tight_layout()
plt.show()

---
## Etapa 9 — Indicadores Fundamentalistas dos 4 Ativos Selecionados
*(Exigência do Edital: Rentabilidade, Valuation e Endividamento)*


In [ ]:
papeis_finais = [t.replace('.SA','') for t in melhor_comb]
df_ind = df[df['Papel'].isin(papeis_finais)][['Papel','ROE','P/L','P/VP','Dív.Brut/ Patrim.']].copy()
df_ind.columns = ['Ativo','ROE (Rentabilidade)','P/L (Valuation)','P/VP (Valuation)','Dív/PL (Endividamento)']
df_ind['ROE (Rentabilidade)'] = df_ind['ROE (Rentabilidade)'].map('{:.1%}'.format)
display(df_ind.reset_index(drop=True))
print('\nIndicadores prontos para inclusão no Relatório Técnico da Competição.')

---
## Conclusão da Estratégia FF5

O modelo Fama-French 5 Fatores é amplamente reconhecido na academia e perfeitamente alinhado ao conteúdo do **Módulo 04** da disciplina (Variáveis de Firma: ROE, ROA, EBITDA, P/L). Ao substituir a triagem subjetiva por um sistema de fatores quantificáveis e replicáveis, eliminamos o viés de seleção e garantimos uma metodologia auditável. A fusão com a otimização combinatória de Markowitz assegura que os **4 ativos** escolhidos são simultaneamente aqueles mais bem posicionados nos 5 fatores **e** a combinação que maximiza o prêmio de risco frente ao CDI.
